<a href="https://colab.research.google.com/github/k-zannnne/mliot-pyml-2026-hw/blob/main/week04/bt_buoi07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

In [ ]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")
df.head()

Đã tải từ seaborn.


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [ ]:
print(df.shape)
print(df.info())
print(df.describe(include='object'))

(891, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB
None
         sex embarked  who  embark_town alive
count    891      

In [ ]:
missing_col=df.isnull().sum()
print(missing_col)

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


In [ ]:
df['age']=df['age'].fillna(df['age'].median())
df['embarked']=df['embarked'].fillna(df['embarked'].mode())
df['fare']=df['fare'].fillna(df['fare'].median())

In [ ]:
df=pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True)

In [ ]:
X=df.drop(columns=['survived'])
y=df['survived']

In [ ]:
X = pd.get_dummies(X, columns=['class', 'who', 'deck', 'embark_town', 'alive'], drop_first=True)
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=42)
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)
log_reg=LogisticRegression()
log_reg.fit(X_train_scaled, y_train)
y_pred_log=log_reg.predict(X_test_scaled)
y_prob_log = log_reg.predict_proba(X_test_scaled)[:, 1]
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)
y_raw_lin = lin_reg.predict(X_test_scaled)
y_pred_lin = (y_raw_lin >= 0.5).astype(int)
def calc_metrics(y_true, y_pred, y_prob=None):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob if y_prob is not None else y_pred)
    }

metrics_log = calc_metrics(y_test, y_pred_log, y_prob_log)
metrics_lin = calc_metrics(y_test, y_pred_lin, y_raw_lin)

df_compare = pd.DataFrame([metrics_lin, metrics_log],
                          index=['Linear Regression (Threshold = 0.5)', 'Logistic Regression'])

print("=== BẢNG SO SÁNH KẾT QUẢ DỰ ĐOÁN ===")
print(df_compare.round(4))

print("\n=== BÁO CÁO PHÂN LOẠI (CLASSIFICATION REPORT) - LOGISTIC REGRESSION ===")
print(classification_report(y_test, y_pred_log))

=== BẢNG SO SÁNH KẾT QUẢ DỰ ĐOÁN ===
                                     Accuracy  Precision  Recall  F1-Score  \
Linear Regression (Threshold = 0.5)       1.0        1.0     1.0       1.0   
Logistic Regression                       1.0        1.0     1.0       1.0   

                                     ROC-AUC  
Linear Regression (Threshold = 0.5)      1.0  
Logistic Regression                      1.0  

=== BÁO CÁO PHÂN LOẠI (CLASSIFICATION REPORT) - LOGISTIC REGRESSION ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       105
           1       1.00      1.00      1.00        74

    accuracy                           1.00       179
   macro avg       1.00      1.00      1.00       179
weighted avg       1.00      1.00      1.00       179



Mô hình dự đoán chính xác khoảng 80% và mô hình Logistic tốt hơn Linear vì đầu ra bài toán là tỉ lệ sống sót [0, 1] trong khi đó Linear cho ra kết quả có giá trị liên tục.

In [38]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [24]:
df=pd.read_excel('Dry_Bean_Dataset.csv')

In [25]:
%pip install openpyxl

In [26]:
df = pd.read_excel(
    'Dry_Bean_Dataset.csv',
    engine="openpyxl"
)

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

print(df.shape)
print(df["class"].head())
print(df["class"].unique())

(13611, 18)
0    SEKER
1    SEKER
2    SEKER
3    SEKER
4    SEKER
Name: class, dtype: object
['SEKER' 'BARBUNYA' 'BOMBAY' 'CALI' 'HOROZ' 'SIRA' 'DERMASON']


In [27]:
target = "class"

numeric_columns = [
    column for column in df.columns
    if column != target
]

df[numeric_columns] = df[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# Làm sạch target dạng chữ
df[target] = (
    df[target]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [28]:
print("Thông tin dữ liệu:")
df.info()

print("\nThống kê mô tả:")
display(df.describe(include="all").T)

Thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13611 entries, 0 to 13610
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bean_id          13611 non-null  int64  
 1   area             13611 non-null  int64  
 2   perimeter        13611 non-null  float64
 3   majoraxislength  13611 non-null  float64
 4   minoraxislength  13611 non-null  float64
 5   aspectration     13611 non-null  float64
 6   eccentricity     13611 non-null  float64
 7   convexarea       13611 non-null  int64  
 8   equivdiameter    13611 non-null  float64
 9   extent           13611 non-null  float64
 10  solidity         13611 non-null  float64
 11  roundness        13611 non-null  float64
 12  compactness      13611 non-null  float64
 13  shapefactor1     13611 non-null  float64
 14  shapefactor2     13611 non-null  float64
 15  shapefactor3     13611 non-null  float64
 16  shapefactor4     13611 non-null  float6

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
bean_id,13611.0,NaN,NaN,NaN,6806.0,3929.301592,1.0,3403.5,6806.0,10208.5,13611.0
area,13611.0,NaN,NaN,NaN,53048.284549,29324.095717,20420.0,36328.0,44652.0,61332.0,254616.0
perimeter,13611.0,NaN,NaN,NaN,855.283459,214.289696,524.736,703.5235,794.941,977.213,1985.37
majoraxislength,13611.0,NaN,NaN,NaN,320.141867,85.694186,183.601165,253.303633,296.883367,376.495012,738.860153
minoraxislength,13611.0,NaN,NaN,NaN,202.270714,44.970091,122.512653,175.84817,192.431733,217.031741,460.198497
aspectration,13611.0,NaN,NaN,NaN,1.583242,0.246678,1.024868,1.432307,1.551124,1.707109,2.430306
eccentricity,13611.0,NaN,NaN,NaN,0.750895,0.092002,0.218951,0.715928,0.764441,0.810466,0.911423
convexarea,13611.0,NaN,NaN,NaN,53768.200206,29774.915817,20684.0,36714.5,45178.0,62294.0,263261.0
equivdiameter,13611.0,NaN,NaN,NaN,253.06422,59.17712,161.243764,215.068003,238.438026,279.446467,569.374358
extent,13611.0,NaN,NaN,NaN,0.749733,0.049086,0.555315,0.718634,0.759859,0.786851,0.866195


In [29]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

display(missing)
print("Tổng số missing values:", df.isna().sum().sum())

,missing_count,missing_percent
bean_id,0,0.0
area,0,0.0
perimeter,0,0.0
majoraxislength,0,0.0
minoraxislength,0,0.0
aspectration,0,0.0
eccentricity,0,0.0
convexarea,0,0.0
equivdiameter,0,0.0
extent,0,0.0


Tổng số missing values: 0


In [30]:
rows_before_missing = len(df)

df = df.dropna().reset_index(drop=True)

rows_after_missing = len(df)

print("Số hàng trước khi bỏ missing:", rows_before_missing)
print("Số hàng sau khi bỏ missing:", rows_after_missing)
print("Số hàng đã bỏ:", rows_before_missing - rows_after_missing)

Số hàng trước khi bỏ missing: 13611
Số hàng sau khi bỏ missing: 13611
Số hàng đã bỏ: 0


In [31]:
duplicate_count = df.duplicated().sum()

print("Số dòng trùng hoàn toàn:", duplicate_count)

df = df.drop_duplicates().reset_index(drop=True)

print("Kích thước sau khi xóa duplicate:", df.shape)

Số dòng trùng hoàn toàn: 0
Kích thước sau khi xóa duplicate: (13611, 18)


In [32]:
target = "class"

# Tất cả các cột ngoại trừ target đều là feature
features = [
    column for column in df.columns
    if column != target
]

X = df[features].copy()
y = df[target].copy()

print("Danh sách feature:")
print(features)

print("\nSố feature:", len(features))
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nPhân bố target:")
print(y.value_counts())

Danh sách feature:
['bean_id', 'area', 'perimeter', 'majoraxislength', 'minoraxislength', 'aspectration', 'eccentricity', 'convexarea', 'equivdiameter', 'extent', 'solidity', 'roundness', 'compactness', 'shapefactor1', 'shapefactor2', 'shapefactor3', 'shapefactor4']

Số feature: 17
X shape: (13611, 17)
y shape: (13611,)

Phân bố target:
class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64


In [33]:
print("Số lượng từng lớp:")
print(y.value_counts().sort_index())

print("\nTỷ lệ từng lớp:")
print(y.value_counts(normalize=True).sort_index().round(4))

print("\nCác nhãn có trong target:")
print(sorted(y.unique()))

Số lượng từng lớp:
class
BARBUNYA    1322
BOMBAY       522
CALI        1630
DERMASON    3546
HOROZ       1928
SEKER       2027
SIRA        2636
Name: count, dtype: int64

Tỷ lệ từng lớp:
class
BARBUNYA    0.0971
BOMBAY      0.0384
CALI        0.1198
DERMASON    0.2605
HOROZ       0.1417
SEKER       0.1489
SIRA        0.1937
Name: proportion, dtype: float64

Các nhãn có trong target:
['BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SEKER', 'SIRA']


In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (10888, 17)
Test shape: (2723, 17)


In [35]:
train_df = X_train.copy()
train_df[target] = y_train

test_df = X_test.copy()
test_df[target] = y_test

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

display(train_df.head())

,bean_id,area,perimeter,majoraxislength,minoraxislength,aspectration,eccentricity,convexarea,equivdiameter,extent,solidity,roundness,compactness,shapefactor1,shapefactor2,shapefactor3,shapefactor4,class
0,1454,42142,741.324,260.523582,206.118244,1.263952,0.611597,42438,231.639506,0.791488,0.993025,0.963627,0.889131,0.006182,0.002383,0.790553,0.999222,SEKER
1,6471,53866,961.806,387.117925,177.923686,2.175753,0.888120,54788,261.886085,0.605168,0.983171,0.731728,0.676502,0.007187,0.000929,0.457655,0.995744,HOROZ
2,4640,74312,1025.211,390.615917,243.809027,1.602139,0.781292,75060,307.598727,0.808442,0.990035,0.888469,0.787471,0.005256,0.001247,0.620111,0.993503,CALI
3,4656,74460,1044.175,407.437813,234.851357,1.734875,0.817160,75603,307.904882,0.792128,0.984882,0.858196,0.755710,0.005472,0.001101,0.571098,0.990783,CALI
4,1937,49108,799.901,278.853164,224.441402,1.242432,0.593447,49455,250.052490,0.784448,0.992984,0.964472,0.896717,0.005678,0.002265,0.804102,0.999042,SEKER


In [36]:
print("Phân bố lớp trong train:")
print(train_df[target].value_counts().sort_index())

print("\nTỷ lệ lớp trong train:")
print(train_df[target].value_counts(normalize=True).sort_index().round(4))

print("\nPhân bố lớp trong test:")
print(test_df[target].value_counts().sort_index())

print("\nTỷ lệ lớp trong test:")
print(test_df[target].value_counts(normalize=True).sort_index().round(4))

Phân bố lớp trong train:
class
BARBUNYA    1057
BOMBAY       418
CALI        1304
DERMASON    2837
HOROZ       1542
SEKER       1621
SIRA        2109
Name: count, dtype: int64

Tỷ lệ lớp trong train:
class
BARBUNYA    0.0971
BOMBAY      0.0384
CALI        0.1198
DERMASON    0.2606
HOROZ       0.1416
SEKER       0.1489
SIRA        0.1937
Name: proportion, dtype: float64

Phân bố lớp trong test:
class
BARBUNYA    265
BOMBAY      104
CALI        326
DERMASON    709
HOROZ       386
SEKER       406
SIRA        527
Name: count, dtype: int64

Tỷ lệ lớp trong test:
class
BARBUNYA    0.0973
BOMBAY      0.0382
CALI        0.1197
DERMASON    0.2604
HOROZ       0.1418
SEKER       0.1491
SIRA        0.1935
Name: proportion, dtype: float64


In [43]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)
log_reg=LogisticRegression()
log_reg.fit(X_train_scaled, y_train)
y_pred_log=log_reg.predict(X_test_scaled)
knn=KNeighborsClassifier()
knn.fit(X_train_scaled, y_train)
y_pred_knn=knn.predict(X_test_scaled)
def evaluate_model(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted')
    rec = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    return {
        'Model': model_name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    }

metrics_log = evaluate_model(y_test, y_pred_log, 'Logistic Regression')
metrics_knn = evaluate_model(y_test, y_pred_knn, 'KNN')
df_results = pd.DataFrame([metrics_log, metrics_knn])
print("BẢNG KẾT QUẢ")
print(df_results.to_string(index=False))

BẢNG KẾT QUẢ
              Model  Accuracy  Precision   Recall  F1-Score
Logistic Regression  0.989717   0.989739 0.989717  0.989696
                KNN  0.983841   0.983810 0.983841  0.983806
